In [ ]:
!pip install mlflow boto3 awscli

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/8

In [ ]:
!aws configure

In [ ]:
import mlflow
mlflow.set_tracking_uri("http://3.237.83.141:5000/")

In [ ]:
mlflow.set_experiment('Exp- Logistic Regression')

<Experiment: artifact_location='s3://mlflow-bucket-rg/6', creation_time=1771460963434, experiment_id='6', last_update_time=1771460963434, lifecycle_stage='active', name='Exp- Logistic Regression', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [ ]:
df = pd.read_csv("/content/reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [ ]:
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=1000,
    min_df=5,
    max_df=0.9
)

In [ ]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# start a mlflow run to track everything inside this block
with mlflow.start_run() as run:

  # adding meta-data to describe this run
  mlflow.set_tag("mlflow.runName","Logistic_Regression_TrainTestSplit")
  mlflow.set_tag("experiment_type", "baseline 2.0")
  mlflow.set_tag("model_type","Logistic Regression")
  mlflow.set_tag("description", "Logistic Regression model for sentiment analysis using Bag of Words and bigrams with simple train test split")

  # adding logs about which vectorizer we used and how big the vocabulary is
  # BoW (CountVectorizer) + bigrams only
  mlflow.log_param("vectorizer_type", "CountVectorizer")
  mlflow.log_param("ngram_range", (1, 2))
  mlflow.log_param("vectorizer_max_features", 1000)
  mlflow.log_param("min_df", 5)
  mlflow.log_param("max_df", 0.9)

 # Logistic Regression hyperparams
  C = 0.3
  penalty = "l2"
  solver = "lbfgs"
  class_weight = "balanced"
  max_iter = 2000

  mlflow.log_param("C", C)
  mlflow.log_param("penalty", penalty)
  mlflow.log_param("solver", solver)
  mlflow.log_param("class_weight", class_weight)
  mlflow.log_param("max_iter", max_iter)

  # train
  model = LogisticRegression(
      C=C,
      penalty=penalty,
      solver=solver,
      class_weight=class_weight,
      max_iter=max_iter,
      random_state=42
  )

  model.fit(X_train,y_train)

  y_pred = model.predict(X_test)

  # compute and log accuracy
  accuracy = accuracy_score(y_test, y_pred)
  mlflow.log_metric("accuracy", accuracy)

  macro_f1 = f1_score(y_test, y_pred, average="macro")
  mlflow.log_metric("macro_f1", macro_f1)

  # generate classification report as a dictionary
  report = classification_report(y_test, y_pred, output_dict=True)

  # this computes precision, recall, f1-score, support for each label and averages it
  for label, metrics in report.items():
    if isinstance(metrics, dict):
      for metric, value in metrics.items():
        mlflow.log_metric(f"{label}_{metric}", value)

  conf_matrix = confusion_matrix(y_test, y_pred)
  plt.figure(figsize=(8,6))
  sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
  plt.xlabel("Predicted")
  plt.ylabel("Actual")
  plt.title("Confusion Matrix")

  # save confusion matrix plot locally and also upload it as artifact
  plt.savefig("confusion_matrix.png")
  mlflow.log_artifact("/content/confusion_matrix.png")

  # Log trained Logistic Regression model to MLflow model registry
  mlflow.sklearn.log_model(model, "logistic_regression_model")

print(f"Accuracy: {accuracy}")

2026/02/25 06:09:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 06:09:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearSVC_TrainTestSplit at: http://3.209.11.118:5000/#/experiments/7/runs/4f8edca7a19b4b09af1826c8a4954541
🧪 View experiment at: http://3.209.11.118:5000/#/experiments/7
Accuracy: 0.7967 | Macro F1: 0.7777


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.6 MB/s eta 0:00:00


In [ ]:
import optuna
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report


EXPERIMENT_NAME = "Exp- Logistic Regression"
mlflow.set_experiment(EXPERIMENT_NAME)


def objective(trial):
    # hyperparams
    C = trial.suggest_categorical("C", [0.03, 0.1, 0.3, 1, 3, 10])
    max_features = trial.suggest_categorical("vectorizer_max_features", [1000, 5000, 10000])

    ngram_range = (1, 2)
    min_df = 5
    max_df = 0.9

    class_weight = "balanced"
    solver = "lbfgs"
    penalty = "l2"
    max_iter = 2000

    run_name = f"Trial_{trial.number}_Logistic_Regression"
    with mlflow.start_run(run_name=run_name, nested=False):

        # log params
        mlflow.log_param("vectorizer_type", "CountVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("min_df", min_df)
        mlflow.log_param("max_df", max_df)
        mlflow.log_param("C", C)
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("class_weight", class_weight)
        mlflow.log_param("solver", solver)
        mlflow.log_param("penalty", penalty)
        mlflow.log_param("max_iter", max_iter)

        model = LogisticRegression(
            C=C,
            penalty=penalty,
            solver=solver,
            class_weight=class_weight,
            max_iter=max_iter,
            random_state=42
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average="macro")

        rep = classification_report(y_test, y_pred, output_dict=True)
        neg_recall = rep.get("-1", {}).get("recall", np.nan)
        neg_f1 = rep.get("-1", {}).get("f1-score", np.nan)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("macro_f1", macro_f1)
        mlflow.log_metric("-1_recall", neg_recall)
        mlflow.log_metric("-1_f1", neg_f1)

        return macro_f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=18)

with mlflow.start_run(run_name="LogReg_Optuna_Summary"):
    mlflow.log_param("best_C", study.best_params["C"])
    mlflow.log_param("best_vectorizer_max_features", study.best_params["vectorizer_max_features"])
    mlflow.log_metric("best_macro_f1", study.best_value)

print("Best params:", study.best_params)
print("Best macro_f1:", study.best_value)

[I 2026-02-26 23:41:49,692] A new study created in memory with name: no-name-0f51ab08-554f-40c1-bae0-c11a2e4fd781
[I 2026-02-26 23:41:56,972] Trial 0 finished with value: 0.8031819950136585 and parameters: {'C': 0.1, 'vectorizer_max_features': 5000}. Best is trial 0 with value: 0.8031819950136585.


🏃 View run Trial_0_Logistic_Regression at: http://3.237.83.141:5000/#/experiments/6/runs/5c33022375d345d38dd5bc01d83a0536
🧪 View experiment at: http://3.237.83.141:5000/#/experiments/6


[I 2026-02-26 23:42:06,977] Trial 1 finished with value: 0.8249129214900949 and parameters: {'C': 10, 'vectorizer_max_features': 1000}. Best is trial 1 with value: 0.8249129214900949.


🏃 View run Trial_1_Logistic_Regression at: http://3.237.83.141:5000/#/experiments/6/runs/acf683e6f5fd4f398a84bbe893b762fb
🧪 View experiment at: http://3.237.83.141:5000/#/experiments/6


[I 2026-02-26 23:42:14,864] Trial 2 finished with value: 0.8249129214900949 and parameters: {'C': 10, 'vectorizer_max_features': 1000}. Best is trial 1 with value: 0.8249129214900949.


...

🏃 View run Trial_15_Logistic_Regression at: http://3.237.83.141:5000/#/experiments/6/runs/55c36e11e9ca497da48dc3a829eda511
🧪 View experiment at: http://3.237.83.141:5000/#/experiments/6


[I 2026-02-26 23:43:53,298] Trial 16 finished with value: 0.7637676235071001 and parameters: {'C': 0.03, 'vectorizer_max_features': 5000}. Best is trial 7 with value: 0.839274027764125.


🏃 View run Trial_16_Logistic_Regression at: http://3.237.83.141:5000/#/experiments/6/runs/68d1fd4408364297a7d6119a13a6e13c
🧪 View experiment at: http://3.237.83.141:5000/#/experiments/6


[I 2026-02-26 23:43:57,552] Trial 17 finished with value: 0.8031819950136585 and parameters: {'C': 0.1, 'vectorizer_max_features': 10000}. Best is trial 7 with value: 0.839274027764125.


🏃 View run Trial_17_Logistic_Regression at: http://3.237.83.141:5000/#/experiments/6/runs/c3dfe2b0e125439da13d9398faca5260
🧪 View experiment at: http://3.237.83.141:5000/#/experiments/6
🏃 View run LogReg_Optuna_Summary at: http://3.237.83.141:5000/#/experiments/6/runs/3b5c99a3e8ec42cfa2420e456785c1ed
🧪 View experiment at: http://3.237.83.141:5000/#/experiments/6
Best params: {'C': 1, 'vectorizer_max_features': 5000}
Best macro_f1: 0.839274027764125
